In [2]:
# Install dependencies (run once)
%pip install google-cloud-bigquery pandas db-dtypes pyarrow

Note: you may need to restart the kernel to use updated packages.


In [3]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT = "realtime-headway-prediction"
client = bigquery.Client(project=PROJECT)
print(f"Connected to project: {client.project}")

Connected to project: realtime-headway-prediction


In [4]:
query = """
select
*
from `headway_prediction.clean`
where route_id in ('A', 'C', 'E')
  and direction = 'S'
"""

df = client.query(query).to_dataframe()
df.head()

,trip_uid,stop_id,track,trip_date,route_id,direction,arrival_time
0,1770973080_A..S57R,A02S,A3,2026-02-13 08:58:00+00:00,A,S,2026-02-13 13:58:20+00:00
1,1769311770_A..S74R,A02S,A3,2026-01-25 03:29:30+00:00,A,S,2026-01-25 08:29:37+00:00
2,1742469930_A..S57R,A02S,A3,2025-03-20 11:25:30+00:00,A,S,2025-03-20 15:25:37+00:00
3,1768679910_A..S,A02S,A3,2026-01-17 19:58:30+00:00,A,S,2026-01-18 00:58:30+00:00
4,1745367090_A..S74X030,A02S,A3,2025-04-23 00:11:30+00:00,A,S,2025-04-23 04:11:52+00:00


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4757706 entries, 0 to 4757705
Data columns (total 7 columns):
 #   Column        Dtype              
---  ------        -----              
 0   trip_uid      object             
 1   stop_id       object             
 2   track         object             
 3   trip_date     datetime64[us, UTC]
 4   route_id      object             
 5   direction     object             
 6   arrival_time  datetime64[us, UTC]
dtypes: datetime64[us, UTC](2), object(5)
memory usage: 254.1+ MB


In [6]:
# create the definitive node ID
df['node_id'] = df['stop_id'] + '_' + df['track']

# if you want to see your absolute vocabulary of nodes
unique_nodes = df['node_id'].unique()

print('Discovered Nodes:', unique_nodes)


Discovered Nodes: ['A02S_A3' nan 'A03S_A3' 'A05S_A3' 'A06S_A3' 'A07S_A3' 'A09S_A3' 'A09S_A1'
 'A10S_A1' 'A11S_A1' 'A12S_A3' 'A12S_A1' 'A14S_A1' 'A15S_A3' 'A15S_A1'
 'A16S_A1' 'A17S_A1' 'A18S_A1' 'A19S_A1' 'A20S_A1' 'A21S_A1' 'A22S_A1'
 'A24S_A3' 'A24S_A1' 'A25S_A1' 'A27S_A3' 'A27S_A1' 'A28S_A3' 'A28S_A1'
 'A30S_A1' 'A31S_A3' 'A31S_A1' 'A32S_A1' 'A32S_A3' 'A33S_A1' 'A34S_A3'
 'A34S_A1' 'A36S_A3' 'A38S_A3' 'A39S_A3' 'A40S_A3' 'A41S_A3' 'A42S_A3'
 'A43S_A1' 'A44S_A1' 'A45S_A1' 'A46S_A3' 'A46S_A1' 'A47S_A1' 'A48S_A3'
 'A48S_A1' 'A49S_A1' 'A50S_A1' 'A51S_A3' 'A51S_A1' 'A52S_A1' 'A53S_A1'
 'A54S_A1' 'A55S_A3' 'A55S_A1' 'A57S_K1' 'A58S_K1' 'A59S_K1' 'A60S_K1'
 'A61S_K1' 'A62S_F3' 'A62S_K1' 'A63S_K1' 'A64S_K1' 'A65S_K1' 'D14S_B3'
 'D21S_B1' 'E01S_A1' 'F14S_B1' 'F16S_B1' 'F18S_B1' 'H02S_F1' 'H03S_F1'
 'H04S_F3' 'H06S_F3' 'H07S_F3' 'H08S_F3' 'H09S_F3' 'H10S_F3' 'H11S_F3'
 'H13S_F3' 'H14S_F3' 'H15S_F3' 'H17S_F3' 'H18S_F3' 'A29S_A1' 'D13S_CM'
 'D17S_B1' 'A25S_D3' 'B06S_T1' 'B08S_T1' 'B10S_B5' 'D14

In [7]:
# sort the dataset to insure time flows correctly
df = df.sort_values(by=['trip_uid','arrival_time'])

# find the next node by shifting the column up by 1
# we group by trip_uid so we don't accidentally connect the end of trip 1 to the start of trip 2
df['next_node_id'] = df.groupby('trip_uid')['node_id'].shift(-1)

print(df[['trip_uid','node_id','next_node_id']])

                   trip_uid  node_id next_node_id
595584   1739768790_A..S43R  A55S_A3      A57S_K1
2244264  1739768790_A..S43R  A57S_K1      A58S_K1
3377507  1739768790_A..S43R  A58S_K1      A59S_K1
3788277  1739768790_A..S43R  A59S_K1      A60S_K1
2138584  1739768790_A..S43R  A60S_K1      A61S_K1
...                     ...      ...          ...
4525172  1771303980_A..S43R  A60S_K1      A61S_K1
4364253  1771303980_A..S43R  A61S_K1      A63S_K1
2700823  1771303980_A..S43R  A63S_K1      A64S_K1
841720   1771303980_A..S43R  A64S_K1      A65S_K1
3879753  1771303980_A..S43R  A65S_K1          NaN

[4757706 rows x 3 columns]


In [8]:
df[['trip_uid','node_id','next_node_id']]

,trip_uid,node_id,next_node_id
595584,1739768790_A..S43R,A55S_A3,A57S_K1
2244264,1739768790_A..S43R,A57S_K1,A58S_K1
3377507,1739768790_A..S43R,A58S_K1,A59S_K1
3788277,1739768790_A..S43R,A59S_K1,A60S_K1
2138584,1739768790_A..S43R,A60S_K1,A61S_K1
...,...,...,...
4525172,1771303980_A..S43R,A60S_K1,A61S_K1
4364253,1771303980_A..S43R,A61S_K1,A63S_K1
2700823,1771303980_A..S43R,A63S_K1,A64S_K1
841720,1771303980_A..S43R,A64S_K1,A65S_K1


In [9]:
# drop the last stops 
edges_df = df.dropna(subset=['next_node_id']).copy()

In [10]:
# count the frequency of every unique (Source -> Target) pair
edge_weights = edges_df.groupby(['node_id','next_node_id']).size().reset_index(name='trip_count')

# calculate a weight as percentage (probability of taking that path)
# e.g. of all trains leaving A02S_Track3, what % go to A03S_track3 vs A03S_track1?
edge_weights['total_departures'] = edge_weights.groupby('node_id')['trip_count'].transform('sum')
edge_weights['weight'] = edge_weights['trip_count'] / edge_weights['total_departures']

print('\nFinal Weighted Edges:')
edge_weights[['node_id','next_node_id','weight']]


Final Weighted Edges:


,node_id,next_node_id,weight
0,A02S_A3,A03S_A3,0.996975
1,A02S_A3,A05S_A3,0.000850
2,A02S_A3,A06S_A3,0.000104
3,A02S_A3,A07S_A3,0.000041
4,A02S_A3,A09S_A3,0.000041
...,...,...,...
2784,Q04S_S1,Q03S_S1,1.000000
2785,R30S_A3,A41S_B1,0.200000
2786,R30S_A3,D22S_B3,0.800000
2787,R31S_F3,R36S_F3,1.000000


In [11]:
# clean the missing data
# drop any rows where eithe the source or the target is missing track info
edge_weights = edge_weights.dropna(subset=['node_id','next_node_id'])
# if your missing tracks showed up as strings like 

# build the node vocabulary
# gather every unique node that appears as either a source or target
all_unique_nodes = pd.concat([
    edge_weights['node_id'],
    edge_weights['next_node_id']
]).unique()

# create the mapping dict: {'AO2S_Track3: 0, 'A03S_Track3':1, ...}
node_to_id = {node: i for i, node in enumerate(all_unique_nodes)}

# map the datafrmae columns to integers
edge_weights['src_id'] = edge_weights['node_id'].map(node_to_id)
edge_weights['tgt_id'] = edge_weights['next_node_id'].map(node_to_id)

# convert to pytorch tensors
# pytorch geometric requires edge_index to be shape [2, num_edges]
source_tensor = torch.tensor(edge_weights['src_id'].values, dtype=torch.long)
target_tensor = torch.tensor(edge_weights['tgt_id'].values, dtype=torch.long)

# stack them together
edge_index = torch.stack([source_tensor, target_tensor], dim=0)

# extract the weights (probabilities) into a 1D tensor
edge_attr = torch.tensor(edge_weights['weight'].values, dtype=torch.float)

print("Edge Index Shape:", edge_index.shape)
print("Edge Attr Shape:", edge_attr.shape)
print("Total unique nodes (graph size):", len(all_unique_nodes))

NameError: name 'torch' is not defined

In [15]:
import pandas as pd
import torch

# 1. Define your threshold
# If an edge (track connection) was used less than 50 times in a year, drop it.
# With ~400 trips a day, 50 is a very safe floor for "this was a freak anomaly."
MIN_TRIPS = 365

# (Assuming 'edges_df' is your chronological dataset from the previous step)
# edges_df has columns: ['trip_uid', 'node_id', 'next_node_id']

# 2. Count the raw frequency of every unique connection
edge_counts = edges_df.groupby(['node_id', 'next_node_id']).size().reset_index(name='trip_count')

# 3. PRUNE THE GRAPH: Drop the ghost tracks and rare reroutes
valid_edges = edge_counts[edge_counts['trip_count'] >= MIN_TRIPS].copy()

# 4. Recalculate weights on the CLEANED dataset
# We group by the source node to find the new total valid departures
valid_edges['total_valid_departures'] = valid_edges.groupby('node_id')['trip_count'].transform('sum')

# Calculate the final probability weight
valid_edges['weight'] = valid_edges['trip_count'] / valid_edges['total_valid_departures']

# 5. Extract the Final Node Vocabulary
# By pulling unique nodes ONLY from the valid_edges dataframe, 
# any "orphan" stations that were only visited during anomalies are automatically dropped.
final_unique_nodes = pd.concat([
    valid_edges['node_id'], 
    valid_edges['next_node_id']
]).unique()

# 6. Create the Golden Dictionary
node_to_id = {node: i for i, node in enumerate(final_unique_nodes)}

# 7. Map strings to integers for PyTorch
valid_edges['src_id'] = valid_edges['node_id'].map(node_to_id)
valid_edges['tgt_id'] = valid_edges['next_node_id'].map(node_to_id)

# 8. Generate the Final Tensors
source_tensor = torch.tensor(valid_edges['src_id'].values, dtype=torch.long)
target_tensor = torch.tensor(valid_edges['tgt_id'].values, dtype=torch.long)

edge_index = torch.stack([source_tensor, target_tensor], dim=0)
edge_attr = torch.tensor(valid_edges['weight'].values, dtype=torch.float)

# Print the final sanity check
print(f"Original unique nodes: {len(pd.concat([edge_counts['node_id'], edge_counts['next_node_id']]).unique())}")
print(f"Final pruned nodes: {len(final_unique_nodes)}")
print(f"Original edges: {len(edge_counts)}")
print(f"Final valid edges: {len(valid_edges)}")

Original unique nodes: 248
Final pruned nodes: 137
Original edges: 2789
Final valid edges: 172


In [18]:
import json

output_path = "../local_artifacts/node_to_id.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(node_to_id, f, indent=2, ensure_ascii=False)

print(f"Saved {len(node_to_id)} items to {output_path}")

Saved 137 items to ../local_artifacts/node_to_id.json


In [ ]:
stops = pd.read_csv('../local_artifacts/raw_data/stops.txt')

In [ ]:
stops[stops.stop_id=='A03S']

### Simulating construction of graph network with tensors and matmul

In [ ]:
path_A = ['59_express','42_express']
path_C = ['59_local','50_local','42_local']
path_E = ['7_ave', '50_local', '42_local']

In [ ]:
# combine all unique stops into a single list

unique_nodes = [
    '59_local',
    '59_express',
    '7_ave',
    '50_local',
    '42_local',
    '42_express'
]

# create a lookup dictionary
node_to_id = {node: i for i, node in enumerate(unique_nodes)}

In [ ]:
source_nodes = []
target_nodes = []

def add_edges_from_path(path):
    # loop through the list, stopping right before the last station
    for i in range(len(path) - 1):
        current_station = path[i]
        next_station = path[i + 1]
    
        # look up their integer IDs
        src_id = node_to_id[current_station]
        tgt_id = node_to_id[next_station]

        source_nodes.append(src_id)
        target_nodes.append(tgt_id)

# build the edges from the distinct routes
add_edges_from_path(path_A)
add_edges_from_path(path_C)
add_edges_from_path(path_E)

print("Sources:", source_nodes)
print("Targets:", target_nodes)




In [ ]:
import torch

# clean up duplicates by converting to a set of tupels then back to lists
unique_edges = list(set(zip(source_nodes, target_nodes)))
src_clean, tgt_clean = zip(*unique_edges)

# create the final edge index tensor required by graph models
edge_index = torch.tensor([src_clean, tgt_clean], dtype=torch.long)

In [ ]:
# matrix shape: (number of nodes, number of feaetures) -> (6, 2)
# mock up state at 8:05 am
x_805am = torch.tensor([
    [0.0, 4.0], # node 0 - 59 local: no train, 4 mins since last C
    [1.0, 0.0], # node 1 - 59 exp: A train is currently here! (0 min)
    [0.0, 2.0], # node 2 - 7 ave: no train, 2 mins since last E left 
    [0.0, 1.0], # node 3 - 50 local: no train, 1 min since last c/e left
    [1.0, 0.0], # node 4 - 42 lcl: C train is currently here! (0 min)
    [0.0, 5.0]  # node 5 - 42 exp: no train, 5 mins since last A left
])

In [ ]:
x_805am

In [ ]:
# Define A and X as shown above
A = torch.tensor([
    [1., 0., 0., 0., 0., 0.],
    [0., 1., 0., 0., 0., 0.],
    [0., 0., 1., 0., 0., 0.],
    [1., 0., 1., 1., 0., 0.],
    [0., 0., 0., 1., 1., 0.],
    [0., 1., 0., 0., 0., 1.]
])

X = torch.tensor([
    [1., 0.],
    [0., 5.],
    [1., 0.],
    [0., 2.],
    [0., 4.],
    [0., 1.]
])

# Perform the message passing step
H = torch.matmul(A, X)
print(H)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Our aggregated matrix (A * X) from the previous step
# Shape: (6, 2)
aggregated_features = torch.tensor([
    [1., 0.],  # Node 0
    [0., 5.],  # Node 1
    [1., 0.],  # Node 2
    [2., 2.],  # Node 3 (The bottleneck)
    [0., 6.],  # Node 4
    [0., 6.]   # Node 5
])

# 2. Define the Weight Matrix (W)
# We want to transform 2 input features into 3 hidden features.
# In PyTorch, nn.Linear is essentially just a wrapper around a Weight Matrix (+ a bias).
# It will initialize with random, learnable numbers.
W = nn.Linear(in_features=2, out_features=3, bias=False)

# 3. Apply the Weight Matrix: (A * X) * W
transformed_features = W(aggregated_features)

# 4. Apply the Activation Function (ReLU)
H_new = F.relu(transformed_features)

print("Shape of H_new:", H_new.shape)
print("\nNew Hidden State for Node 3 (50th St):")
print(H_new[3].detach().numpy())

In [ ]:
import torch

# Minute 1 (8:00 AM)
X_1 = torch.tensor([
    [0., 2.], [0., 4.], [0., 1.], [0., 5.], [1., 0.], [0., 2.]
])

# Minute 2 (8:01 AM) - The train at Node 4 leaves. Headways increase.
X_2 = torch.tensor([
    [0., 3.], [0., 5.], [0., 2.], [0., 6.], [0., 1.], [0., 3.]
])

# Minute 3 (8:02 AM) - A C train arrives at Node 0!
X_3 = torch.tensor([
    [1., 0.], [0., 6.], [0., 3.], [0., 7.], [0., 2.], [0., 4.]
])

# Stack them along a new dimension (dim=0 for Time)
# This creates our (T, N, F) tensor
X_Sequence = torch.stack([X_1, X_2, X_3], dim=0)

print("Sequence Shape (T, N, F):", X_Sequence.shape)
# Output: torch.Size([3, 6, 2])

# If we want to isolate the 3-minute history of JUST Node 3 (50th St):
# We select all time steps (:), Node index 3, and all features (:)
node_3_history = X_Sequence[:, 3, :]
print("\nHistory of Node 3 (50th St):")
print(node_3_history.numpy())

In [ ]:
stops[stops.stop_id=='A02S']